# CellularComponent → Gene Relation Pipeline

Builds a unified, deduplicated edge table for the **CellularComponent–Gene** relation  
by ingesting processed files from multiple KG sources, normalising gene identifiers  
via NCBI/Ensembl mapping dictionaries, and writing the final triple table to disk.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- PrimeKG
- Phenknowlator

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Mapping databases: `/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [1]:
import pandas as pd

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CELLULARCOMPONENT_GENE/ALL_CELLCOMPONENT_GENE.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name'
]

## 1 · Load Gene Identifier Mapping Dictionaries

Two mapping resources are loaded:
1. **Ensembl → NCBI Gene Symbol** (`ENS_2NCBI`): used to attach Ensembl IDs to NCBI symbols.
2. **NCBI gene_info** (`Homo_sapiens.gene_info`): used to resolve GeneID ↔ Symbol ↔ Description.

From these, four lookup dictionaries are built for downstream normalisation:
- `NCBI_2ENS_dict` – NCBI Symbol → Ensembl ID
- `NCBI_ID2sym_dict` – GeneID → Symbol
- `NCBI_ID2desc_dict` – GeneID → Description
- `NCBI_sym2desc_dict` – Symbol → Description
- `NCBI_syn2sym_dict` – exploded Synonym → canonical Symbol

In [2]:
# ── Ensembl ↔ NCBI Gene Symbol ────────────────────────────────────────────────
ens2ncbi = pd.read_csv(MAPPING_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ens2ncbi = ens2ncbi[~ens2ncbi['name'].isna()]          # drop rows without a gene symbol
NCBI_2ENS_dict = dict(zip(ens2ncbi['name'], ens2ncbi['initial_alias']))
del ens2ncbi

# ── NCBI gene_info (human) ────────────────────────────────────────────────────
ncbi = pd.read_csv(MAPPING_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
ncbi['ENSEMBLE_ID'] = ncbi['Symbol'].map(NCBI_2ENS_dict)

NCBI_ID2sym_dict  = dict(zip(ncbi['GeneID'],   ncbi['Symbol']))
NCBI_ID2desc_dict = dict(zip(ncbi['GeneID'],   ncbi['description']))
NCBI_sym2desc_dict = dict(zip(ncbi['Symbol'],  ncbi['description']))

# Explode pipe-separated synonyms into individual keys → canonical symbol
NCBI_syn2sym_dict = {}
for synonyms, symbol in zip(ncbi['Synonyms'], ncbi['Symbol']):
    for syn in str(synonyms).split('|'):
        NCBI_syn2sym_dict[syn.strip()] = symbol

print(f"NCBI genes loaded: {len(ncbi):,}")
print(f"Synonym entries:   {len(NCBI_syn2sym_dict):,}")

NCBI genes loaded: 193,505
Synonym entries:   69,564


## 2 · Load Source Files

Each source DataFrame is loaded, columns lowercased, and `tail_id_is` / `head_id_is`  
set to the appropriate ID namespace.

### 2.1 PrimeKG

In [3]:
PrimeKG_CELLCOMP_GENE = pd.read_csv(PROC_DIR + 'PrimeKG/PrimekG_CellularComp_Gene.csv')
PrimeKG_CELLCOMP_GENE.columns = PrimeKG_CELLCOMP_GENE.columns.str.lower()
PrimeKG_CELLCOMP_GENE['tail_id_is'] = 'NCBI_ID'
PrimeKG_CELLCOMP_GENE['kg_type'] = 'Generalised'
print(f"PrimeKG CellComp→Gene: {len(PrimeKG_CELLCOMP_GENE):,} rows")
PrimeKG_CELLCOMP_GENE.head(3)

PrimeKG CellComp→Gene: 83,402 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_alias_mapped,tail_detail_name,tail_ens,kg_type
0,GO:1904813,CellularComponent_Gene,A1BG,CellularComponent,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,A1BG,alpha-1-B glycoprotein,ENSG00000121410,Generalised
1,GO:1904813,CellularComponent_Gene,ACLY,CellularComponent,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,ACLY,ATP citrate lyase,ENSG00000131473,Generalised
2,GO:1904813,CellularComponent_Gene,AGL,CellularComponent,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,AGL,"amylo-alpha-1,6-glucosidase and 4-alpha-glucan...",ENSG00000162688,Generalised


## 2.2 Pheknowlator

In [4]:
Pheknowlator_CELLCOMP_GENE = pd.read_csv(PROC_DIR + 'pheknowlator/CellularComponent_Gene.csv')
Pheknowlator_CELLCOMP_GENE.columns = Pheknowlator_CELLCOMP_GENE.columns.str.lower()
Pheknowlator_CELLCOMP_GENE['tail_id_is'] = 'NCBI_ID'
Pheknowlator_CELLCOMP_GENE['head_type'] = 'CellularComponent'
Pheknowlator_CELLCOMP_GENE['relation'] = 'CellularComponent_Gene'
Pheknowlator_CELLCOMP_GENE['kg_type'] = 'Generalised'

print(f"PrimeKG CellComp→Gene: {len(Pheknowlator_CELLCOMP_GENE):,} rows")
Pheknowlator_CELLCOMP_GENE.head(3)

PrimeKG CellComp→Gene: 172,407 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,kg_source,tail_id_is,kg_type
0,GO:0005788,CellularComponent_Gene,COL5A2,CellularComponent,Gene,endoplasmic reticulum lumen,collagen type V alpha 2 chain,Quick_GO,pheknowlator,NCBI_ID,Generalised
1,GO:0032587,CellularComponent_Gene,RAC1,CellularComponent,Gene,ruffle membrane,Rac family small GTPase 1,Quick_GO,pheknowlator,NCBI_ID,Generalised
2,GO:0005634,CellularComponent_Gene,CREBRF,CellularComponent,Gene,nucleus,CREB3 regulatory factor,Quick_GO,pheknowlator,NCBI_ID,Generalised


## 3 · Consolidate into Unified Schema

All source DataFrames are aligned to `REQUIRED_COLS` (missing columns filled with `pd.NA`),  
then concatenated into `final_df`.

In [11]:
# List all source DataFrames to include
source_dfs = [
    PrimeKG_CELLCOMP_GENE,
    Pheknowlator_CELLCOMP_GENE
    # ← add further source DataFrames here
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 255,809


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,GO:1904813,CellularComponent_Gene,A1BG,CellularComponent,interacts with,Gene,PrimeKG,Generalised,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,alpha-1-B glycoprotein
1,GO:1904813,CellularComponent_Gene,ACLY,CellularComponent,interacts with,Gene,PrimeKG,Generalised,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,ATP citrate lyase
2,GO:1904813,CellularComponent_Gene,AGL,CellularComponent,interacts with,Gene,PrimeKG,Generalised,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,"amylo-alpha-1,6-glucosidase and 4-alpha-glucan..."
3,GO:1904813,CellularComponent_Gene,ALAD,CellularComponent,interacts with,Gene,PrimeKG,Generalised,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,aminolevulinate dehydratase
4,GO:1904813,CellularComponent_Gene,LOC112694756,CellularComponent,interacts with,Gene,PrimeKG,Generalised,Quick_GO,NCBI_ID,ficolin-1-rich granule lumen,uncharaterized LOC112694756
...,...,...,...,...,...,...,...,...,...,...,...,...
255804,GO:0005788,CellularComponent_Gene,FKBP14,CellularComponent,None,Gene,pheknowlator,Generalised,Quick_GO,NCBI_ID,endoplasmic reticulum lumen,FKBP prolyl isomerase 14
255805,GO:0005739,CellularComponent_Gene,MMUT,CellularComponent,None,Gene,pheknowlator,Generalised,Quick_GO,NCBI_ID,mitochondrion,methylmalonyl-CoA mutase
255806,GO:0005925,CellularComponent_Gene,PFN1,CellularComponent,None,Gene,pheknowlator,Generalised,Quick_GO,NCBI_ID,focal adhesion,profilin 1
255807,GO:0005654,CellularComponent_Gene,AKR1B1,CellularComponent,None,Gene,pheknowlator,Generalised,Quick_GO,NCBI_ID,nucleoplasm,aldo-keto reductase family 1 member B


In [17]:
final_df['species'] = "Homo sapiens"

## 4 · Sanity Check — Distinct Values

In [18]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'CellularComponent_Gene'}
head_type           : {'CellularComponent'}
tail_type           : {'Gene'}
relation_type       : {None, 'interacts with'}
kg_source           : {'pheknowlator', 'PrimeKG'}
head_id_is          : {'Quick_GO'}
tail_id_is          : {'NCBI_ID'}


## 5 · Gene Name Normalisation (tail)

Rows where `tail_detail_name` is missing are resolved using:
1. Strip hyphens from the tail gene symbol.
2. Map through the synonym dictionary to get the canonical NCBI symbol.
3. Map canonical symbol to its description via `NCBI_sym2desc_dict`.
4. Drop rows where `tail_detail_name` remains unresolved.

In [19]:
# Step 1–2: resolve synonyms for rows with missing tail_detail_name
mask = final_df['tail_detail_name'].isna()
final_df.loc[mask, 'tail'] = (
    final_df.loc[mask, 'tail']
    .str.replace('-', '', regex=False)
    .map(NCBI_syn2sym_dict)
    .fillna(final_df.loc[mask, 'tail'])
)

# Step 3: fill description for still-missing rows
mask = final_df['tail_detail_name'].isna()
final_df.loc[mask, 'tail_detail_name'] = final_df.loc[mask, 'tail'].map(NCBI_sym2desc_dict)

# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 255,754 remaining


## 6 · NaN Audit (pre-dedup)

In [20]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,172353,0,172353
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


## 7 · Deduplication

Group on `(head, relation, tail)`. For `kg_source`, merge all unique sources with `::` delimiter.  
All other metadata columns take the first non-null value.

In [21]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 255,754  |  After dedup: 89,080


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0000015,CellularComponent_Gene,ENO1,CellularComponent,interacts with,Gene,PrimeKG::pheknowlator,Quick_GO,NCBI_ID,phosphopyruvate hydratase complex,enolase 1,Generalised,Homo sapiens
1,GO:0000015,CellularComponent_Gene,ENO2,CellularComponent,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,phosphopyruvate hydratase complex,enolase 2,Generalised,Homo sapiens
2,GO:0000015,CellularComponent_Gene,ENO3,CellularComponent,interacts with,Gene,PrimeKG,Quick_GO,NCBI_ID,phosphopyruvate hydratase complex,enolase 3,Generalised,Homo sapiens


## 8 · Post-dedup NaN Audit & Source Distribution

In [22]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,6881,0,6881
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [23]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'pheknowlator', 'PrimeKG::pheknowlator', 'PrimeKG'} kg_source
PrimeKG::pheknowlator    50570
PrimeKG                  31629
pheknowlator              6881
Name: count, dtype: int64


## 9 · Save Output

In [24]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 89,080 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CELLULARCOMPONENT_GENE/ALL_CELLCOMPONENT_GENE.csv
